# MuJoCo URDF Import
Onshape can export assemblies in the [Universal Robotic Description Format (URDF)](https://wiki.ros.org/urdf). MuJoCo only has limited support for working with these files directly [(docs section)](https://mujoco.readthedocs.io/en/stable/modeling.html#curdf), so instead parse some key information from it to create a MuJoCo-native MJCF XML file.

## Config

In [ ]:
from pathlib import Path

import mujoco as mj

ONSHAPE_ASSEMBLY = "assembly_bala2ish"

mass_override = {
	# "electronics": 0.032
	"wheel": 0.017,
	"wheel_1": 0.017,
	"chassis": 0.127,
	"sensor_imu": 0,
}

# Onshape-exported geometry of wheels is not ideal, use cylinders instead

wheelsize = [45/1000/2, 14.15/1000/2]
geom_override = {
	# "wheel": dict(
	# 	type=mj.mjtGeom.mjGEOM_CYLINDER,
	# 	mass=0.025,
	# 	size=wheelsize,
	# 	zaxis=[1, 0, 0],
	# 	pos=[wheelsize[0]/4, 0, 0]
	# ),
	# "wheel_1": dict(
	# 	type=mj.mjtGeom.mjGEOM_CYLINDER,
	# 	mass=0.025,
	# 	size=[45/1000/2, 14.15/1000/2],
	# 	zaxis=[1, 0, 0],
	# 	pos=[-wheelsize[0]/4, 0, 0]
	# ),
}

cwd = Path(".")
path_assembly = cwd / ONSHAPE_ASSEMBLY
path_urdf = path_assembly / "urdf" / (ONSHAPE_ASSEMBLY + ".urdf")
path_meshes = path_assembly / "meshes"

In [70]:
def mapl(items, func):
	return list(map(func, items))

def parse_vec(string):
	return mapl(string.split(" "), lambda x: float(x))

## Parsing

In [71]:
import xml.etree.ElementTree as ET


xml = ET.parse(path_urdf)

robot = xml.getroot()

links = {}

print("Links (bodies):")
for link in robot.findall("./link"):
	name = link.get("name")
	print(f"  {name}")
	links[name] = {
		"xml": link,
		"children": []
	}

print()
print("Joints:")
for joint in robot.findall("./joint"):
	joint_name = joint.get("name")
	joint_type = joint.get("type")
	if not (joint_type == "fixed" or joint_type == "continuous"):
		print(f"  '{joint_name}' unsupported mate type: {joint_type}")
		continue
	parent = joint.find("./parent").get("link")
	child = joint.find("./child").get("link")
	print(f"  '{joint_name}' {joint_type} {child} to {parent}")
	p = links.get(parent)
	mate = {
		"joint_name": joint_name,
		"joint_type": joint_type,
		"joint_axis": parse_vec(joint.find("./axis").get("xyz")),
		"origin": parse_vec(joint.find("./origin").get("xyz")),
		"child": child
	}
	p["children"].append(mate)


Links (bodies):
  chassis
  root
  sensor_imu
  wheel
  wheel_1

Joints:
  'fastened_1' fixed chassis to root
  'fastened_2' fixed sensor_imu to chassis
  'wheel_left' continuous wheel_1 to chassis
  'wheel_right' continuous wheel to chassis


In [76]:
spec = mj.MjSpec.from_file("template.xml")

spec.compiler.meshdir = str(path_meshes)


def convert(mate, parent=spec.worldbody):
	name = mate.get("child")
	link = links.get(name)
	print(name, mate, link)
	
	inertial = link["xml"].find("./inertial")
	body: mj.MjsBody = parent.add_body(
		name=name,
		pos=mate["origin"],
		# explicitinertial=parent is not spec.worldbody,
	)
	mesh = link["xml"].find("./visual/geometry/mesh")
	color = link["xml"].find("./visual/material/color")
	if color is not None:
		print("  ", "color", color)
		color = parse_vec(color.get("rgba"))

	# if parent is spec.worldbody:
	# 	# root body
	# 	body.add_site(
	# 		name="site_root",
	# 		pos=[0, 0, geom_override["wheel"]["size"][0]],
	# 		size=[0.001, 0.001, 0.001],
	# 	)

	if mesh is not None:
		if inertial is None:
			print(f"error: mesh {name} has no inertial info")
			return

		mass = float(inertial.find("mass").get("value"))

		# origin = inertial.find("origin")
		# in_pos = parse_vec(origin.get("xyz"))
		# in_orient = parse_vec(origin.get("rpy"))

		# inertia = inertial.find("inertia")
		# inermat = []
		# for dim in ["ixx", "iyy", "izz", "ixy", "ixz", "iyz"]:
		# 	inermat.append(float(inertia.get(dim)))

		# body.mass=mass_override.get(name, mass)
		# body.position=in_pos,
		# body.orientation=in_orient,
		# body.fullinertia=inermat

		mesh_file = Path(mesh.get("filename").replace("package://", "")).relative_to(
			path_meshes
		)
		mesh_name = name
		spec.add_mesh(
			name=mesh_name, file=str(mesh_file), scale=parse_vec(mesh.get("scale"))
		)
		spec.assets[mesh_name] = str(mesh_file)
		geom_name = f"geom_{name}"
		pos = parse_vec(link["xml"].find("./visual/origin").get("xyz"))
		body.add_geom(
			name=geom_name,
			rgba=color,

			type=mj.mjtGeom.mjGEOM_MESH,
			meshname=mesh_name,
			mass=mass_override.get(name, mass),
			pos=pos,
			**geom_override.get(name, dict())
		)

	else:
		print(f"  {name} has no geometry")

	joint_name = mate["joint_name"]
	match mate["joint_type"]:
		case "freejoint":
			body.add_freejoint(name=joint_name)
		case "fixed":
			pass  # mujoco default
		case "continuous" if not joint_name.startswith("wheel_back"):
			body.add_joint(
				name=f"joint_{joint_name}",
				type=mj.mjtJoint.mjJNT_HINGE,
				axis=mate["joint_axis"],
				damping=0.001,
				frictionloss=0.001,
				# damping=0,
				# frictionloss=0,
				#  armature: motor's rotational inertia reflected through the gearbox
				#            Estimated rotational inertia of N20 motor: 5e-8 Kg-m^2.
				#            With a 1:N gearbox, we multiply the inertia by N^2 to get
				#            the reflected inertia. So, 5e-8 * 30^2 = 4.5e-5 Kg-m^2.
				#            This is ~10x the wheel's inertia, so this dominates. -->
				#
				armature=4.5e-5,
				# armature=0,
			)

			#  Control torque (for a hinge joint) is determined by gear * ctrl (units
			#  of N-m). For reinforcement learning, it helps to have a normalized
			#  ctrl, so we set the limits to [-1, 1]. With the gear set to 0.05, that
			#  gives us actual limits of -0.05 to 0.05 N-m, which is a decent estimate
			#  of our N20 motor with a 1:30 gearbox.

			act = spec.add_actuator(name=f"act_{joint_name}")
			# act.damping = [0, 0, 0] # messes with dcmotor..
			# act.set_to_dcmotor(
			# 	# measured going ~1m/0.73s, 43.6mm dia wheels, = 600rpm (motor spec 500rpm at 6V, but running at 8V)
			# 	# at 30:1 gearing so shaft = 18000 rpm
			# 	# so motor constant Kv = 0.004244
			# 	# motorconst=[0.004244, 0.004244],
			# 	# motorconst=[0.00382, 0.00382], # 6V, 500rpm
			# 	# motorconst=[0.004, 0.004],
			# 	# motorconst=[0.16, 0.16],
			# 	motorconst=[0, 0],
			# 	# resistance=11.5,  # ohm
			# 	resistance=0,
			# 	nominal=[
			# 		8, 0.15*8/11.5, 8.023*2*math.pi
			# 	],
			# 	saturation=[
			# 		0,
			# 		0.15,
			# 		0,
			# 	],
			# 	inductance=[
			# 		0.0026,  # 2.6mH based on https://www.klikrobotics.com/PWMTutorial2.html
			# 		0,
			# 	],
			# 	thermal=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
			# 	cogging=[0.0, 0.0, 0.0]
			# )
			# # act.gear[0] = 30 # 30:1 gearbox
			# act.ctrlrange = [-8, 8]

			act.set_to_motor()
			act.ctrlrange = [-1, 1]
			# # act.gear[0] = 0.05 # original
			# # act.gear[0] = 0.0058 # 1:1
			# # act.gear[0] = 0.007 # fast gearing
			# # act.gear[0] = 0.031 # fastest gearing
			# # new 500rpm motors
			act.gear[0] = 0.05

			act.target = f"joint_{joint_name}"
			act.trntype = mj.mjtTrn.mjTRN_JOINT

			spec.add_sensor(
				name=f"jointvel_{joint_name}",
				type=mj.mjtSensor.mjSENS_JOINTVEL,
				objtype=mj.mjtObj.mjOBJ_JOINT,
				objname=f"joint_{joint_name}",
			)

	if name.startswith("sensor_"):
		body.add_site(name=f"site_{name}", pos=[0, 0, 0], size=[0.001, 0.001, 0.001])
		sensor_type = name.replace("sensor_", "")
		if sensor_type == "imu":
			# spec.add_sensor(name="imu_accel", type=mj.mjtSensor.mjSENS_ACCELEROMETER, site="site_sensor_imu")
			# spec.add_sensor(name="imu_gyro", type=mj.mjtSensor.mjSENS_GYRO, site="site_sensor_imu")
			# spec.add_sensor(name="imu_orientation", objtype=mj.MjsSite, objname="site_sensor_imu")

			# BNO055 provides orientation quaternion
			spec.add_sensor(
				name="imu_quat",
				type=mj.mjtSensor.mjSENS_FRAMEQUAT,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
				# reftype=mj.mjtObj.mjOBJ_SITE,
				# refname="site_sensor_imu"
			)
			spec.add_sensor(
				name="imu_accel",
				type=mj.mjtSensor.mjSENS_ACCELEROMETER,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)
			spec.add_sensor(
				name="imu_gyro",
				type=mj.mjtSensor.mjSENS_GYRO,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)

	for c in link["children"]:
		convert(c, body)


convert(
	{
		"joint_name": "_root",
		"joint_type": "freejoint",
		# "origin": [0,0,0.5],
		"origin": [0, 0, 0],
		"child": "root",
	}
)

# privileged velocity info
spec.add_sensor(
	name="priv_vel",
	type=mj.mjtSensor.mjSENS_VELOCIMETER,
	objtype=mj.mjtObj.mjOBJ_SITE,
	objname="site_sensor_imu",
)

print()
print(spec.to_xml())

spec.encode("robot.xml")


root {'joint_name': '_root', 'joint_type': 'freejoint', 'origin': [0, 0, 0], 'child': 'root'} {'xml': <Element 'link' at 0x6ffedc9e42c0>, 'children': [{'joint_name': 'fastened_1', 'joint_type': 'fixed', 'joint_axis': [-0.0, -0.0, -1.0], 'origin': [0.0, 0.0, 0.0], 'child': 'chassis'}]}
  root has no geometry
chassis {'joint_name': 'fastened_1', 'joint_type': 'fixed', 'joint_axis': [-0.0, -0.0, -1.0], 'origin': [0.0, 0.0, 0.0], 'child': 'chassis'} {'xml': <Element 'link' at 0x6ffedc9e6c00>, 'children': [{'joint_name': 'fastened_2', 'joint_type': 'fixed', 'joint_axis': [0.0, 0.0, -1.0], 'origin': [0.008, 0.0175, 0.0625], 'child': 'sensor_imu'}, {'joint_name': 'wheel_left', 'joint_type': 'continuous', 'joint_axis': [1.0, 3.20732e-48, 2.1684e-16], 'origin': [-0.031, 1.70406e-32, 0.0225], 'child': 'wheel_1'}, {'joint_name': 'wheel_right', 'joint_type': 'continuous', 'joint_axis': [1.0, 3.20732e-48, 2.1684e-16], 'origin': [0.031, 8.86823e-50, 0.0225], 'child': 'wheel'}]}
   color <Element 'co

2789

In [ ]:
import mujoco as mj

for name in ["robot", "robot-bala2-mj"]:
	spec = mj.MjSpec.from_file(f"{name}.xml")
	model = spec.compile()
	data = mj.MjData(model)

	mj.mj_forward(model, data)

	robot = data.body("root")

	com = robot.subtree_com * 1000

	axlepos = (data.body("wheel").xpos + data.body("wheel_1").xpos)/2

	print(data.body("wheel").xpos, data.body("wheel_1").xpos)

	print(name)
	print(f"  mass {model.body("root").subtreemass[0]*1000:.4f} gram")
	print(f"  center of mass [ {com[0]:.4f} {com[1]:.4f} {com[2]:.4f} ] mm")
	print(f"  axle position {axlepos[2]*1000:.4f} mm [ {axlepos[0]:.4f} {axlepos[1]:.4f} {axlepos[2]:.4f} ]")
	print(f"  pendulum arm {(robot.subtree_com[2] - data.body("wheel").xpos[2])*1000:.4f} mm")

	if name == "robot":
		site = data.site("site_sensor_imu")
	elif name == "robot-bala2-mj":
		site = data.site("imu_site")
	relpos = (site.xpos - axlepos)*1000
	print(f"  IMU pos relative to axle [ {relpos[0]:.4f} {relpos[1]:.4f} {relpos[2]:.4f} ] mm")

	print()

robot
  mass 161.0000 gram
  center of mass [ 0.0898 0.0000 41.0202 ] mm
  axle position 22.5000 mm [ 0.0075 0.0000 0.0225 ]
  pendulum arm 18.5202 mm
  IMU pos relative to axle [ 0.5000 17.5000 40.0000 ] mm

robot-bala2-mj
  mass 161.0000 gram
  center of mass [ 0.0000 0.0000 35.9099 ] mm
  axle position 22.5000 mm [ 0.0000 0.0000 0.0225 ]
  pendulum arm 13.4099 mm
  IMU pos relative to axle [ -8.0000 22.5000 40.0000 ] mm



# Launch MuJoCo

In [82]:
import mujoco as mj
import mujoco.viewer

spec = mj.MjSpec.from_file("robot.xml")
model = spec.compile()
data = mj.MjData(model)

mujoco.viewer.launch(model, data)

In [42]:
import time

import mujoco as mj
import mujoco.viewer as mjv

from types import SimpleNamespace
from scipy.spatial.transform import Rotation
import numpy as np

spec = mj.MjSpec.from_file("robot-bala2-mj.xml")
model = spec.compile()
data = mj.MjData(model)

try:
	viewer.close()
except:
	pass

step = 0

a1 = data.actuator("act_wheel_left")
a2 = data.actuator("act_wheel_right")
jv1 = data.sensor("jointvel_wheel_left")
jv2 = data.sensor("jointvel_wheel_right")
robot = data.body("root")

d = SimpleNamespace(
			robot=data.body("root"),  # to get position
			act1=data.actuator("act_wheel_left"),
			act2=data.actuator("act_wheel_right"),
			imu_gyro=data.sensor("imu_gyro"),
			imu_quat=data.sensor("imu_quat"),
			priv_vel=data.sensor("priv_vel"),
			input="voltage"
		)


# a1.ctrl = -1
a2.ctrl = 1

mj.mj_forward(model, data)

print("robot mass", model.body("root").subtreemass)
print("robot center of mass", robot.subtree_com)
print("axle position", data.body("wheel").xpos[2])
print("pendulum arm", robot.subtree_com[2] - data.body("wheel").xpos[2])
vels = []

viewer = mjv.launch_passive(model, data)

while True:
	step += 1
	step_start = time.time()
	mj.mj_step(model, data)
	viewer.sync()

	gyro = d.imu_gyro.data
	quat = d.imu_quat.data

	r = Rotation.from_quat(quat, scalar_first=True)
	r.inv()

	eul = r.as_euler("xyz", degrees=False, suppress_warnings=True)
	if step % 30 == 0:
		print(step, "vel", jv1.data[0], jv2.data[0])
		# print(step, "eul", eul)
		# print(" " * (len(f"{step}")), "vel", d.priv_vel.data)


	# if step == 400:
	# 	pos = robot.xpos[1]
	# if step > 400:
	# 	vels.append(jv1.data[0])
	# if step > 800:
	# 	print("moved", (robot.xpos[1] - pos)/2, "m/s")
	# 	print("avg vel steps 400-800", np.mean(vels)/(2*np.pi), "rotations per second")
	# 	viewer.close()
	# 	time.sleep(1)
	# 	break

	# print(step, j1.qvel, j1.qacc)

	sleep = 0.005 - (time.time() - step_start)
	if sleep > 0:
		time.sleep(sleep)


robot mass [0.161]
robot center of mass [0.         0.         0.03590994]
axle position 0.0225
pendulum arm 0.013409937888198747
30 vel 0.0 46.02400465208289
60 vel 0.0 48.83589646403248
90 vel 0.0 48.99095093662162
120 vel 0.0 48.99950101289687
150 vel 0.0 48.99997248465188
180 vel 0.0 48.99999848273757
210 vel 0.0 48.9999999163345
240 vel 0.0 48.99999999538648
270 vel 0.0 48.999999999745604
300 vel 0.0 48.999999999985974
330 vel 0.0 48.999999999999226
360 vel 0.0 48.999999999999964
390 vel 0.0 48.999999999999964
420 vel 0.0 48.999999999999964
450 vel 0.0 48.999999999999964
480 vel 0.0 48.999999999999964
510 vel 0.0 48.999999999999964
540 vel 0.0 48.999999999999964
570 vel 0.0 48.999999999999964
600 vel 0.0 48.999999999999964
630 vel 0.0 48.999999999999964
660 vel 0.0 48.999999999999964
690 vel 0.0 48.999999999999964
720 vel 0.0 48.999999999999964
750 vel 0.0 48.999999999999964
780 vel 0.0 48.999999999999964
810 vel 0.0 48.999999999999964
840 vel 0.0 48.999999999999964
870 vel 0.0 48

KeyboardInterrupt: 

In [44]:
viewer.close()